In [1]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

data = pd.read_csv('exoTrain.csv')

X = data.drop('LABEL', axis=1).values
y = data['LABEL'].values - 1  # make labels 0/1

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)

X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
X_val   = torch.tensor(X_val, dtype=torch.float32).unsqueeze(1)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)

train_dataset = TensorDataset(X_train, y_train)
val_dataset   = TensorDataset(X_val, y_val)

class_counts = torch.bincount(y_train)
class_weights = 1.0 / class_counts.float()
sample_weights = class_weights[y_train]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)
val_loader   = DataLoader(val_dataset, batch_size=32)

class CNN_LSTM(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=5),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=5),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=128,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)

model = CNN_LSTM()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# TRAIN 
epochs = 15

for epoch in range(epochs):
    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        outputs = model(xb)
        loss = criterion(outputs, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}")

#  VALIDATION 
def evaluate(loader, name):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for xb, yb in loader:
            outputs = model(xb)
            _, preds = torch.max(outputs, 1)

            all_preds.append(preds)
            all_labels.append(yb)

    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    print(f"\n{name} RESULTS")
    print("Accuracy:", (all_preds == all_labels).float().mean().item())
    print(confusion_matrix(all_labels.numpy(), all_preds.numpy()))
    print(classification_report(all_labels.numpy(), all_preds.numpy()))

# Validate
evaluate(val_loader, "VALIDATION")

test_data = pd.read_csv('exoTest.csv')

X_test = test_data.drop('LABEL', axis=1).values
y_test = test_data['LABEL'].values - 1

X_test = scaler.transform(X_test)

X_test = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)
y_test = torch.tensor(y_test, dtype=torch.long)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=32
)

evaluate(test_loader, "TEST")

Epoch 1, Train Loss: 85.4106
Epoch 2, Train Loss: 75.2884
Epoch 3, Train Loss: 70.9494
Epoch 4, Train Loss: 63.3672
Epoch 5, Train Loss: 57.5964
Epoch 6, Train Loss: 65.9859
Epoch 7, Train Loss: 57.4505
Epoch 8, Train Loss: 47.5444
Epoch 9, Train Loss: 29.2091
Epoch 10, Train Loss: 21.9378
Epoch 11, Train Loss: 28.1206
Epoch 12, Train Loss: 25.9636
Epoch 13, Train Loss: 13.6080
Epoch 14, Train Loss: 8.6820
Epoch 15, Train Loss: 11.3125

VALIDATION RESULTS
Accuracy: 0.7966601252555847
[[808 203]
 [  4   3]]
              precision    recall  f1-score   support

           0       1.00      0.80      0.89      1011
           1       0.01      0.43      0.03         7

    accuracy                           0.80      1018
   macro avg       0.50      0.61      0.46      1018
weighted avg       0.99      0.80      0.88      1018


TEST RESULTS
Accuracy: 0.8438596725463867
[[478  87]
 [  2   3]]
              precision    recall  f1-score   support

           0       1.00      0.85      0